# Facecat Kronos — Walk-Forward Backtest (Colab GPU)

Run the full 4-layer screener backtest with **GPU-accelerated XGBoost** on Colab.

**Prerequisites:**
- Upload `data/ohlcv_all_a.pkl` and `data/benchmark_000905.pkl` to your Google Drive under `MyDrive/kronos/data/`.
- Or adjust the paths in the Config cell below.

**Runtime:** Select *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# Cell 1: Setup — install dependencies and mount Drive
!pip install -q xgboost baostock pandas numpy scipy scikit-learn

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/kronos'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# Cell 2: Clone repo and add to path
import sys

REPO_DIR = '/content/facecat-kronos'

if not os.path.exists(REPO_DIR):
    # Option A: Clone from GitHub (update URL to your repo)
    # !git clone https://github.com/youruser/facecat-kronos.git {REPO_DIR}

    # Option B: Copy from Drive
    !cp -r /content/drive/MyDrive/kronos/facecat-kronos {REPO_DIR}

sys.path.insert(0, REPO_DIR)
print(f'Repo at: {REPO_DIR}')
print(f'sys.path[0]: {sys.path[0]}')

In [ ]:
# Cell 3: Config — GPU XGBoost override
from screener.config import ScreenerConfig

cfg = ScreenerConfig(
    ohlcv_pickle_path=os.path.join(DRIVE_ROOT, 'data/ohlcv_all_a.pkl'),
    benchmark_pickle_path=os.path.join(DRIVE_ROOT, 'data/benchmark_000905.pkl'),
    drive_root=os.path.join(DRIVE_ROOT, 'output/screener'),
)

# Enable GPU for XGBoost (Colab T4)
cfg.layer1_xgb_params['device'] = 'cuda'
cfg.layer2_xgb_params['device'] = 'cuda'

print('XGBoost device (Layer 1):', cfg.layer1_xgb_params.get('device'))
print('XGBoost device (Layer 2):', cfg.layer2_xgb_params.get('device'))
print(f'Train: {cfg.train_start} → {cfg.train_end}')
print(f'Backtest: {cfg.backtest_start} → {cfg.backtest_end}')
print(f'Run ID: {cfg.run_id}')
print(f'Run dir: {cfg.run_dir}')
print(f'Cache dir: {cfg.cache_dir}')

In [ ]:
# Cell 4: Run backtest
import time
from screener.backtester import WalkForwardBacktester

bt = WalkForwardBacktester(cfg)

t0 = time.time()
results = bt.run(run_kronos=False)
elapsed = time.time() - t0

print(f'\nTotal wall time: {elapsed/60:.1f} min')

In [ ]:
# Cell 5: Display results and save to Drive
import pandas as pd
from google.colab import files

# Metrics
print('=== Backtest Metrics ===')
for k, v in results['metrics'].items():
    print(f'  {k:>20}: {v:.4f}' if isinstance(v, float) else f'  {k:>20}: {v}')

# NAV curve
nav = results['nav_series']
if nav is not None and len(nav) > 0:
    nav_s = pd.Series(nav) if not isinstance(nav, pd.Series) else nav
    nav_s.plot(title='Portfolio NAV', figsize=(12, 4))

# Save results + config to run dir
bt.save_results(results)

# Also save models
bt.layer1.save()
bt.layer2.save()

print(f'\nRun folder: {cfg.run_dir}')
print('Contents:')
for f in sorted(os.listdir(cfg.run_dir)):
    full = os.path.join(cfg.run_dir, f)
    if os.path.isdir(full):
        print(f'  {f}/')
        for sf in sorted(os.listdir(full)):
            print(f'    {sf}')
    else:
        print(f'  {f}')

# Download results pickle
results_path = os.path.join(cfg.run_dir, 'backtest_results.pkl')
print(f'\nDownloading {results_path}…')
files.download(results_path)

In [ ]:
# Cell 6: List past runs
runs_dir = os.path.join(cfg.drive_root, 'runs')
if os.path.isdir(runs_dir):
    runs = sorted(os.listdir(runs_dir), reverse=True)
    print(f'Past runs ({len(runs)}):')
    for r in runs:
        run_path = os.path.join(runs_dir, r)
        contents = os.listdir(run_path)
        print(f'  {r}  ({", ".join(sorted(contents))})')
else:
    print('No runs yet.')